In [90]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

In [79]:
data = joblib.load('data/one_slay.joblib')
data

,theta,h,k_up,k_down,k_vals,Re(r),Im(r)
0,0.174533,"(0.1,)",1,1,"(0.1,)",0.002418,-0.050144
1,0.698132,"(0.1,)",1,1,"(0.1,)",0.000772,-0.064522
2,0.872665,"(0.1,)",1,1,"(0.1,)",-0.000976,-0.076848
3,0.349066,"(0.1,)",1,1,"(0.1,)",0.002170,-0.052568
4,1.047198,"(0.1,)",1,1,"(0.1,)",-0.004816,-0.098521
...,...,...,...,...,...,...,...
383995,1.047198,"(3.0000000000000004,)",4,2,"(1.0,)",-0.466667,-0.884433
383996,1.047198,"(3.0000000000000004,)",4,4,"(1.0,)",-0.466667,-0.884433
383997,1.396263,"(3.0000000000000004,)",4,4,"(1.0,)",-0.935672,-0.352871
383998,1.221730,"(3.0000000000000004,)",4,4,"(1.0,)",-0.750447,-0.660930


In [80]:
n_layers = 1
cols_h = sum([[f"h{i}"] for i in range(1, n_layers + 1)], [])
data[cols_h] = pd.DataFrame(data["h"].tolist(), index=data.index)

cols_k = sum([[f"k{i}"] for i in range(1, n_layers + 1)], [])
data[cols_k] = pd.DataFrame(data["k_vals"].tolist(), index=data.index)

In [81]:
data

,theta,h,k_up,k_down,k_vals,Re(r),Im(r),h1,k1
0,0.174533,"(0.1,)",1,1,"(0.1,)",0.002418,-0.050144,0.1,0.1
1,0.698132,"(0.1,)",1,1,"(0.1,)",0.000772,-0.064522,0.1,0.1
2,0.872665,"(0.1,)",1,1,"(0.1,)",-0.000976,-0.076848,0.1,0.1
3,0.349066,"(0.1,)",1,1,"(0.1,)",0.002170,-0.052568,0.1,0.1
4,1.047198,"(0.1,)",1,1,"(0.1,)",-0.004816,-0.098521,0.1,0.1
...,...,...,...,...,...,...,...,...,...
383995,1.047198,"(3.0000000000000004,)",4,2,"(1.0,)",-0.466667,-0.884433,3.0,1.0
383996,1.047198,"(3.0000000000000004,)",4,4,"(1.0,)",-0.466667,-0.884433,3.0,1.0
383997,1.396263,"(3.0000000000000004,)",4,4,"(1.0,)",-0.935672,-0.352871,3.0,1.0
383998,1.221730,"(3.0000000000000004,)",4,4,"(1.0,)",-0.750447,-0.660930,3.0,1.0


In [82]:
X = data.drop(columns=['h', 'k_vals', 'Re(r)', 'Im(r)']+cols_k)
y = data['k1']

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [84]:
for model in [LinearRegression]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model())
    ])

    pipe.fit(X_train, y_train)
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)    

    train_mae, train_rmse = mean_absolute_error(y_train, y_pred_train), np.sqrt(mean_squared_error(y_train, y_pred_train))
    print(f"MAE: {train_mae:.4f}\tRMSE: {train_rmse:.4f}")

    test_mae, test_rmse = mean_absolute_error(y_test, y_pred_test), np.sqrt(mean_squared_error(y_test, y_pred_test))
    print(f"MAE: {test_mae:.4f}\tRMSE: {test_rmse:.4f}")

MAE: 0.2270	RMSE: 0.2622
MAE: 0.2282	RMSE: 0.2634


In [85]:
def extract_features(r):
    R = np.abs(r)
    phi = np.unwrap(np.angle(r))
    R2 = np.abs(r)**2
    return {
        "Abs(r)" : R,
        "R2" : R2,
        "Angle(r)" : phi,
        "min(r)" : np.min(np.real(R2)),
        "max(r)" : np.max(np.real(R2)),
    }    

In [86]:
new_features = extract_features(data['Re(r)'] + 1j*data['Im(r)'])
X = pd.concat([X, pd.DataFrame(new_features)], axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [92]:
for model in [LinearRegression]:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model())
    ])

    pipe.fit(X_train, y_train)
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)    

    train_mae, train_rmse = mean_absolute_error(y_train, y_pred_train), np.sqrt(mean_squared_error(y_train, y_pred_train))
    print(f"Train:\tMAE: {train_mae:.4f}\tRMSE: {train_rmse:.4f}")

    test_mae, test_rmse = mean_absolute_error(y_test, y_pred_test), np.sqrt(mean_squared_error(y_test, y_pred_test))
    print(f"Test:\tMAE: {test_mae:.4f}\tRMSE: {test_rmse:.4f}")

Train:	MAE: 0.2195	RMSE: 0.2559
Test:	MAE: 0.2207	RMSE: 0.2572
